In [ ]:
# ============================================================
# 04_XGBoost.ipynb
# Section 1 — Setup, Paths, and Data Loading
# Purpose: Load preprocessed FD001 data, import XGBoost,
#          verify everything for modeling.
# ============================================================

# --- 1a. Mount Drive ---

# --- 1b. Canonical paths (same as all notebooks) ---
import os

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
DATA_DIR      = os.path.join(PROJECT_ROOT, "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
FIGURES_DIR   = os.path.join(PROJECT_ROOT, "figures")
EXPERIMENT_DIR = os.path.join(PROJECT_ROOT, "experiments")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")

import sys
sys.path.insert(0, SRC_DIR)

# --- 1c. Imports (xgboost is preinstalled in Colab) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from preprocessing import FINAL_FEATURES_FD001

print(f"XGBoost version: {xgb.__version__}")

# --- 1d. Load preprocessed data ---
train = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))

print("\n=== Data loaded from 02_Preprocessing output ===\n")
print(f"Train shape:     {train.shape}")
print(f"Test-last shape: {test_last.shape}")

# --- 1e. Verification (same checks as Notebook 03) ---
assert train.shape[0] == 20631
assert test_last.shape[0] == 100
assert all(f in train.columns for f in FINAL_FEATURES_FD001)
assert "RUL_linear" in train.columns
assert "RUL_piecewise125" in train.columns
assert "RUL_true" in test_last.columns

print("✓ All checks passed. Ready for XGBoost.")

In [ ]:
# ============================================================
# 04_XGBoost.ipynb
# Section 2 — Evaluation Metrics (identical to Notebook 03)
# Purpose: MAE, RMSE, NASA Score (Saxena et al. 2008), and the
#          evaluate_model helper. Same code as Notebook 03.
# ============================================================

def nasa_score(y_true, y_pred):
    """
    PHM08 asymmetric scoring function (Saxena et al. 2008).
    d = predicted - true
    d < 0 (early, conservative): exp(-d/13) - 1
    d >= 0 (late, dangerous):    exp( d/10) - 1
    Lower is better. Penalizes late predictions more heavily.
    """
    d = y_pred - y_true
    scores = np.where(d < 0,
                      np.exp(-d / 13.0) - 1.0,
                      np.exp( d / 10.0) - 1.0)
    return float(np.sum(scores))


def evaluate_model(y_true, y_pred, model_name="Model", label_name="label"):
    """Compute MAE, RMSE, NASA Score; print summary; return dict."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)

    print(f"--- {model_name} | {label_name} ---")
    print(f"  MAE:        {mae:.2f} cycles")
    print(f"  RMSE:       {rmse:.2f} cycles")
    print(f"  NASA Score: {ns:.2f}")
    print()

    return {
        "model": model_name,
        "rul_label": label_name,
        "mae": round(mae, 2),
        "rmse": round(rmse, 2),
        "nasa_score": round(ns, 2),
    }


# --- Sanity checks (same as Notebook 03) ---
perfect = nasa_score(np.array([100, 50, 25]), np.array([100, 50, 25]))
assert perfect == 0.0
early_penalty = nasa_score(np.array([100]), np.array([90]))   # d = -10
late_penalty  = nasa_score(np.array([100]), np.array([110]))  # d = +10
assert late_penalty > early_penalty

print("✓ Metrics defined and verified (identical to Notebook 03).")

In [ ]:
# ============================================================
# 04_XGBoost.ipynb
# Section 3 — Validation Split, Bounded Tuning, Final Evaluation
# Purpose: Engine-level validation split; documented 6-config
#          sweep; select on validation; final train on all
#          engines; evaluate once on test. Both RUL labels.
# ============================================================

# --- 3a. Engine-level validation split (fixed seed) ---
rng = np.random.RandomState(42)
all_units = train["unit"].unique()
val_units = rng.choice(all_units, size=20, replace=False)
train_units = np.array([u for u in all_units if u not in val_units])

train_sub = train[train["unit"].isin(train_units)]
val_sub   = train[train["unit"].isin(val_units)]

# Validation protocol mimics the test protocol: last cycle per engine.
val_last = val_sub.loc[val_sub.groupby("unit")["cycle"].idxmax()]

print(f"Training engines:   {len(train_units)}  ({len(train_sub)} rows)")
print(f"Validation engines: {len(val_units)}  (evaluated at last cycle, n={len(val_last)})")
print(f"Validation units:   {sorted(val_units.tolist())}\n")

X_train_sub = train_sub[FINAL_FEATURES_FD001].values
X_val_last  = val_last[FINAL_FEATURES_FD001].values

# NOTE on validation targets: at a training engine's true last cycle
# RUL = 0, which is trivial. To make validation informative, we instead
# truncate each validation engine at a RANDOM cycle (like test engines)
# and validate at that cycle. Fixed seed => reproducible.
def make_truncated_validation(val_sub, rng, min_keep=30):
    """Truncate each validation engine at a random cycle (>= min_keep),
    return last-row-per-engine DataFrame with true RUL attached."""
    rows = []
    for unit, grp in val_sub.groupby("unit"):
        grp = grp.sort_values("cycle")
        c_max = grp["cycle"].max()
        cut = rng.randint(min_keep, c_max)     # random truncation point
        row = grp[grp["cycle"] == cut].iloc[0].copy()
        row["RUL_val_true"] = c_max - cut       # true remaining cycles
        rows.append(row)
    return pd.DataFrame(rows)

val_eval = make_truncated_validation(val_sub, np.random.RandomState(123))
X_val = val_eval[FINAL_FEATURES_FD001].values
y_val_true = val_eval["RUL_val_true"].values

print(f"Truncated validation set: {len(val_eval)} engines, "
      f"RUL range {y_val_true.min()}–{y_val_true.max()}\n")

# --- 3b. The documented sweep: 6 configurations ---
SWEEP = [
    dict(max_depth=4, n_estimators=300, learning_rate=0.05),
    dict(max_depth=6, n_estimators=300, learning_rate=0.05),
    dict(max_depth=8, n_estimators=300, learning_rate=0.05),
    dict(max_depth=4, n_estimators=600, learning_rate=0.03),
    dict(max_depth=6, n_estimators=600, learning_rate=0.03),
    dict(max_depth=8, n_estimators=600, learning_rate=0.03),
]
FIXED = dict(subsample=0.8, colsample_bytree=0.8,
             random_state=42, n_jobs=-1, objective="reg:squarederror")

def run_sweep(y_col):
    """Train each sweep config on train_sub, evaluate RMSE on the
    truncated validation set. Returns list of (config, val_rmse)."""
    y_tr = train_sub[y_col].values
    results = []
    for cfg in SWEEP:
        model = xgb.XGBRegressor(**cfg, **FIXED)
        model.fit(X_train_sub, y_tr)
        pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val_true, pred))
        results.append((cfg, rmse))
        print(f"  depth={cfg['max_depth']}, n_est={cfg['n_estimators']}, "
              f"lr={cfg['learning_rate']}  ->  val RMSE {rmse:.2f}")
    return results

print("=== Sweep — linear label ===")
sweep_linear = run_sweep("RUL_linear")
best_cfg_linear = min(sweep_linear, key=lambda t: t[1])[0]
print(f"Best (linear): {best_cfg_linear}\n")

print("=== Sweep — piecewise125 label ===")
sweep_pw = run_sweep("RUL_piecewise125")
best_cfg_pw = min(sweep_pw, key=lambda t: t[1])[0]
print(f"Best (piecewise): {best_cfg_pw}\n")

# --- 3c. Final training on ALL 100 engines with best configs ---
X_train_full = train[FINAL_FEATURES_FD001].values
X_test  = test_last[FINAL_FEATURES_FD001].values
y_true  = test_last["RUL_true"].values

# Run 1 — linear label, best config
start_time = time.time()
xgb_linear = xgb.XGBRegressor(**best_cfg_linear, **FIXED)
xgb_linear.fit(X_train_full, train["RUL_linear"].values)
train_time_linear = time.time() - start_time
y_pred_linear = xgb_linear.predict(X_test)

result_linear = evaluate_model(y_true, y_pred_linear,
                               model_name="XGBoost (tuned)",
                               label_name="linear")
result_linear["training_time_sec"] = round(train_time_linear, 4)
result_linear["config"] = str(best_cfg_linear)

# Run 2 — piecewise label, best config
start_time = time.time()
xgb_piecewise = xgb.XGBRegressor(**best_cfg_pw, **FIXED)
xgb_piecewise.fit(X_train_full, train["RUL_piecewise125"].values)
train_time_pw = time.time() - start_time
y_pred_piecewise = xgb_piecewise.predict(X_test)

result_piecewise = evaluate_model(y_true, y_pred_piecewise,
                                  model_name="XGBoost (tuned)",
                                  label_name="piecewise125")
result_piecewise["training_time_sec"] = round(train_time_pw, 4)
result_piecewise["config"] = str(best_cfg_pw)

# --- 3d. Summary vs. Linear Regression baseline ---
LR_BASELINE = {
    "linear":       {"mae": 25.64, "rmse": 32.20, "nasa": 14093.94, "time": 0.0894},
    "piecewise125": {"mae": 17.79, "rmse": 22.07, "nasa": 1334.57,  "time": 0.0239},
}

print("=" * 72)
print("SUMMARY — XGBoost (tuned) vs. Linear Regression")
print("=" * 72)
print(f"{'':<20} {'LR linear':>11} {'XGB linear':>11} {'LR piece':>11} {'XGB piece':>11}")
print("-" * 72)
print(f"{'MAE':<20} {LR_BASELINE['linear']['mae']:>11.2f} {result_linear['mae']:>11.2f} "
      f"{LR_BASELINE['piecewise125']['mae']:>11.2f} {result_piecewise['mae']:>11.2f}")
print(f"{'RMSE':<20} {LR_BASELINE['linear']['rmse']:>11.2f} {result_linear['rmse']:>11.2f} "
      f"{LR_BASELINE['piecewise125']['rmse']:>11.2f} {result_piecewise['rmse']:>11.2f}")
print(f"{'NASA Score':<20} {LR_BASELINE['linear']['nasa']:>11.2f} {result_linear['nasa_score']:>11.2f} "
      f"{LR_BASELINE['piecewise125']['nasa']:>11.2f} {result_piecewise['nasa_score']:>11.2f}")
print(f"{'Train time (s)':<20} {LR_BASELINE['linear']['time']:>11.4f} {result_linear['training_time_sec']:>11.4f} "
      f"{LR_BASELINE['piecewise125']['time']:>11.4f} {result_piecewise['training_time_sec']:>11.4f}")
print("=" * 72)

mae_change  = 100 * (result_piecewise["mae"]  - LR_BASELINE["piecewise125"]["mae"])  / LR_BASELINE["piecewise125"]["mae"]
rmse_change = 100 * (result_piecewise["rmse"] - LR_BASELINE["piecewise125"]["rmse"]) / LR_BASELINE["piecewise125"]["rmse"]
time_ratio  = result_piecewise["training_time_sec"] / LR_BASELINE["piecewise125"]["time"]

print(f"\nXGBoost tuned (piecewise) vs LR (piecewise):")
print(f"  MAE change:  {mae_change:+.1f}%")
print(f"  RMSE change: {rmse_change:+.1f}%")
print(f"  Training-time ratio: {time_ratio:.0f}x LR")
print(f"  Chosen config: {best_cfg_pw}")

In [ ]:
# ============================================================
# 04_XGBoost.ipynb
# Section 4 — Visualization, Zone Analysis, Bias Test
# Purpose: Diagnostic figures; zone analysis vs LR baseline;
#          identify extreme outliers behind the linear-label
#          NASA Score explosion.
# ============================================================

# --- 4a. Predicted vs True scatter — both labels ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_pred, label_name, res in [
    (axes[0], y_pred_linear,    "Linear RUL label",        result_linear),
    (axes[1], y_pred_piecewise, "Piecewise-125 RUL label", result_piecewise),
]:
    ax.scatter(y_true, y_pred, alpha=0.6, s=35, color="#2E75B6",
               edgecolor="white", linewidth=0.5)
    lims = [0, max(y_true.max(), y_pred.max()) * 1.05]
    ax.plot(lims, lims, "r--", linewidth=1.2, label="Perfect prediction")
    ax.set_xlabel("True RUL (cycles)")
    ax.set_ylabel("Predicted RUL (cycles)")
    ax.set_title(f"XGBoost (tuned) — {label_name}\n"
                 f"MAE {res['mae']:.2f} | RMSE {res['rmse']:.2f} | "
                 f"NASA {res['nasa_score']:.0f}")
    ax.legend(loc="upper left")
    ax.set_xlim(lims)
    ax.set_ylim(lims)

fig.suptitle("FD001 — XGBoost: predicted vs. true RUL "
             "(100 test engines, last cycle)", fontsize=13, y=1.02)
fig.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "fd001_xgb_pred_vs_true.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}\n")

# --- 4b. Error vs True RUL ---
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, label_name in [
    (axes2[0], y_pred_linear,    "Linear RUL label"),
    (axes2[1], y_pred_piecewise, "Piecewise-125 RUL label"),
]:
    errors = y_pred - y_true
    ax.scatter(y_true, errors, alpha=0.6, s=35, color="#D9822B",
               edgecolor="white", linewidth=0.5)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("True RUL (cycles)")
    ax.set_ylabel("Prediction error (pred − true, cycles)")
    ax.set_title(f"{label_name}\nAbove 0 = late (dangerous), below 0 = early (conservative)")

fig2.suptitle("FD001 — XGBoost: prediction error vs. true RUL", fontsize=13, y=1.03)
fig2.tight_layout()
fig2_path = os.path.join(FIGURES_DIR, "fd001_xgb_error_vs_true.png")
fig2.savefig(fig2_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig2_path}\n")

# --- 4c. Zone analysis (piecewise) — the bias test vs LR ---
print("=== Zone analysis — XGBoost piecewise vs. LR baseline ===\n")
errors_pw = y_pred_piecewise - y_true

# LR baseline zone values (recorded, Notebook 03)
LR_ZONES = {"Near failure (RUL <= 30)":  (19.85, +18.77),
            "Mid life (30 < RUL <= 80)": (19.76, +19.75),
            "Early life (RUL > 80)":     (16.14, -8.46)}

zones = [("Near failure (RUL <= 30)",  y_true <= 30),
         ("Mid life (30 < RUL <= 80)", (y_true > 30) & (y_true <= 80)),
         ("Early life (RUL > 80)",     y_true > 80)]

print(f"{'Zone':<28} {'n':>4} {'XGB MAE':>9} {'XGB bias':>10} {'LR MAE':>8} {'LR bias':>9}")
print("-" * 74)
for zone_name, mask in zones:
    n = mask.sum()
    zone_mae  = np.abs(errors_pw[mask]).mean()
    zone_bias = errors_pw[mask].mean()
    lr_mae, lr_bias = LR_ZONES[zone_name]
    print(f"{zone_name:<28} {n:>4} {zone_mae:>9.2f} {zone_bias:>+10.2f} "
          f"{lr_mae:>8.2f} {lr_bias:>+9.2f}")

# --- 4d. Outlier hunt: what exploded the linear-label NASA Score? ---
print("\n=== Top 5 NASA-Score contributors — linear label ===\n")
d = y_pred_linear - y_true
scores = np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)
top5 = np.argsort(scores)[::-1][:5]

print(f"{'unit':>5} {'true RUL':>9} {'predicted':>10} {'error':>8} {'score contrib':>14}")
print("-" * 52)
for i in top5:
    print(f"{test_last['unit'].values[i]:>5} {y_true[i]:>9.0f} "
          f"{y_pred_linear[i]:>10.2f} {d[i]:>+8.2f} {scores[i]:>14.1f}")
print(f"\nTop-5 share of total NASA Score: "
      f"{100 * scores[top5].sum() / scores.sum():.1f}%")

In [ ]:
# ============================================================
# 04_XGBoost.ipynb
# Section 5 — Feature Importance
# Purpose: Which sensors does XGBoost rely on? Cross-check
#          against EDA degradation-signal findings.
# ============================================================

# --- 5a. Extract gain-based importance (piecewise model — the strong one) ---
importance = xgb_piecewise.get_booster().get_score(importance_type="gain")

# Map XGBoost's internal feature names (f0, f1, ...) back to sensor names
feature_map = {f"f{i}": name for i, name in enumerate(FINAL_FEATURES_FD001)}
importance_named = {feature_map.get(k, k): v for k, v in importance.items()}

# Sort descending
imp_sorted = sorted(importance_named.items(), key=lambda x: x[1], reverse=True)

print("=== XGBoost feature importance (gain) — piecewise model ===\n")
print(f"{'Rank':>4} {'Sensor':>8} {'Gain':>12} {'Share %':>9}")
print("-" * 38)
total_gain = sum(v for _, v in imp_sorted)
for rank, (feat, gain) in enumerate(imp_sorted, 1):
    print(f"{rank:>4} {feat:>8} {gain:>12.1f} {100*gain/total_gain:>8.1f}%")

# --- 5b. Bar chart ---
fig, ax = plt.subplots(figsize=(10, 6))
feats = [f for f, _ in imp_sorted]
gains = [g for _, g in imp_sorted]
ax.barh(range(len(feats)), gains, color="#2E75B6", edgecolor="white")
ax.set_yticks(range(len(feats)))
ax.set_yticklabels(feats)
ax.invert_yaxis()  # most important on top
ax.set_xlabel("Importance (gain)")
ax.set_title("FD001 — XGBoost feature importance (piecewise-125 label)")
fig.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "fd001_xgb_feature_importance.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {fig_path}")

# --- 5c. Cross-check note ---
print("\n=== Cross-check against EDA ===")
print("Top-5 XGBoost sensors:", [f for f, _ in imp_sorted[:5]])
print("(Compare with EDA Section 12: sensors s9, s14 diverged most;")
print(" Section 7: strongest trends in s2,s3,s4,s8,s9,s11,s13,s15,s17.)")

In [ ]:
# ============================================================
# 04_XGBoost.ipynb
# Section 6 — Save Predictions, Sweep Table, and Finalize
# Purpose: Save per-engine predictions + tuning sweep to results/,
#          print final summary for the experiment log.
# ============================================================

# --- 6a. Per-engine predictions table ---
predictions = pd.DataFrame({
    "unit": test_last["unit"].values,
    "cycle_last": test_last["cycle"].values,
    "RUL_true": y_true,
    "pred_XGB_linear": np.round(y_pred_linear, 2),
    "pred_XGB_piecewise125": np.round(y_pred_piecewise, 2),
})
predictions["error_linear"] = np.round(predictions["pred_XGB_linear"] - predictions["RUL_true"], 2)
predictions["error_piecewise125"] = np.round(predictions["pred_XGB_piecewise125"] - predictions["RUL_true"], 2)

print("Per-engine predictions (first 10 rows):")
print(predictions.head(10).to_string(index=False))

pred_path = os.path.join(RESULTS_DIR, "fd001_xgb_predictions.csv")
predictions.to_csv(pred_path, index=False)
print(f"\nSaved predictions to:\n{pred_path}")

# --- 6b. Save the tuning sweep table (both labels) ---
sweep_rows = []
for cfg, rmse in sweep_linear:
    sweep_rows.append({**cfg, "label": "linear", "val_rmse": round(rmse, 2)})
for cfg, rmse in sweep_pw:
    sweep_rows.append({**cfg, "label": "piecewise125", "val_rmse": round(rmse, 2)})
sweep_df = pd.DataFrame(sweep_rows)

# Mark the selected config per label
sweep_df["selected"] = False
for label, best in [("linear", best_cfg_linear), ("piecewise125", best_cfg_pw)]:
    mask = ((sweep_df["label"] == label) &
            (sweep_df["max_depth"] == best["max_depth"]) &
            (sweep_df["n_estimators"] == best["n_estimators"]) &
            (sweep_df["learning_rate"] == best["learning_rate"]))
    sweep_df.loc[mask, "selected"] = True

sweep_path = os.path.join(RESULTS_DIR, "fd001_xgb_tuning_sweep.csv")
sweep_df.to_csv(sweep_path, index=False)
print(f"\nSaved tuning sweep to:\n{sweep_path}\n")
print("Tuning sweep table:")
print(sweep_df.to_string(index=False))

# --- 6c. Final summary block ---
print("\n" + "=" * 60)
print("NOTEBOOK 04 COMPLETE — Final Results Summary")
print("=" * 60)
print(f"""
Model: XGBoost (tuned via bounded 6-config sweep on 20 held-out engines)
Features: 13 sensors (final set from Notebook 02)
Best config (linear):    {best_cfg_linear}
Best config (piecewise): {best_cfg_pw}
Test protocol: last available cycle per test engine (n=100)

Run 1 — linear label:
  MAE {result_linear['mae']}, RMSE {result_linear['rmse']}, NASA {result_linear['nasa_score']}, time {result_linear['training_time_sec']}s

Run 2 — piecewise125 label:
  MAE {result_piecewise['mae']}, RMSE {result_piecewise['rmse']}, NASA {result_piecewise['nasa_score']}, time {result_piecewise['training_time_sec']}s

Files saved:
  results/fd001_xgb_predictions.csv
  results/fd001_xgb_tuning_sweep.csv
  figures/fd001_xgb_pred_vs_true.png
  figures/fd001_xgb_error_vs_true.png
  figures/fd001_xgb_feature_importance.png
""")
print("=" * 60)

# Notebook 04 — XGBoost: Complete Documentation

## Purpose

Train and evaluate the second model of the complexity ladder: XGBoost,
a gradient-boosted decision tree ensemble. The model is trained twice —
once with the linear RUL label and once with the piecewise-125 label —
using a bounded, documented tuning sweep with a proper engine-level
validation split. Evaluated on the same 100 test engines at their last
available cycle as Notebook 03.

**Position in the project.** Second of four modeling notebooks
(03 Linear Regression → 04 XGBoost → 05 LSTM → 06 Chronos). All four use
identical frozen data from `02_Preprocessing.ipynb` and the identical
evaluation protocol.

**Why XGBoost as step 2.** Linear Regression (Notebook 03) revealed three
structural weaknesses: (1) a globally linear sensor→RUL assumption
contradicted by the exponential degradation signal in the EDA; (2) no
feature interactions (cannot represent "if A is high AND B is low");
(3) regression to the mean producing a dangerous +18.77-cycle
over-prediction bias in the near-failure zone (RUL ≤ 30). XGBoost
addresses (1) and (2) via non-linear decision trees and tree-path
AND-conditions. Whether it reduces (3) is the testable hypothesis of
this notebook.

## Inputs and outputs

**Inputs (from `data/processed/`):**
- `train_FD001_processed.parquet` (20631 rows, 13 features + 2 RUL labels)
- `test_FD001_last_cycle.parquet` (100 rows, 13 features + RUL_true)

**Outputs:**
- `results/fd001_xgb_predictions.csv` — per-engine predictions, both labels
- `results/fd001_xgb_tuning_sweep.csv` — 12-row sweep table with selected flag
- `figures/fd001_xgb_pred_vs_true.png` — predicted vs. true scatter
- `figures/fd001_xgb_error_vs_true.png` — error structure diagnostic
- `figures/fd001_xgb_feature_importance.png` — gain-based importance bar chart
- Experiment log rows EXP-003 (linear) and EXP-004 (piecewise125)

## Section overview

**Section 1 — Setup and data loading.** Same frozen data and paths as all
modeling notebooks. XGBoost 3.3.0 confirmed.

**Section 2 — Evaluation metrics.** Identical NASA Score, MAE, and RMSE
functions as Notebook 03 — same code, same sanity checks, guaranteed
consistency.

**Section 3 — Validation split, bounded tuning, final evaluation.**
Engine-level validation split: 20 of the 100 training engines held out
(random seed 42), validated at a random truncation cycle (seed 123) to
mimic the real test protocol. Six configurations tested, varying max_depth
(4, 6, 8) and n_estimators/learning_rate (300/0.05, 600/0.03). Fixed
parameters: subsample=0.8, colsample_bytree=0.8, random_state=42.
Best config selected on validation RMSE; model retrained on all 100
engines; test set touched once per label.

**Sweep results (validation RMSE, linear label):**

| max_depth | n_estimators | learning_rate | val RMSE |
|---|---|---|---|
| 4 | 300 | 0.05 | 26.78 |
| 6 | 300 | 0.05 | 29.09 |
| **8** | **300** | **0.05** | **25.78 ← selected** |
| 4 | 600 | 0.03 | 26.53 |
| 6 | 600 | 0.03 | 27.34 |
| 8 | 600 | 0.03 | 27.40 |

**Sweep results (validation RMSE, piecewise label):**

| max_depth | n_estimators | learning_rate | val RMSE |
|---|---|---|---|
| 4 | 300 | 0.05 | 25.83 |
| 6 | 300 | 0.05 | 26.22 |
| **8** | **300** | **0.05** | **25.03 ← selected** |
| 4 | 600 | 0.03 | 25.38 |
| 6 | 600 | 0.03 | 25.57 |
| 8 | 600 | 0.03 | 26.85 |

**Section 4 — Visualization and zone analysis.** Predicted-vs-true scatter,
error-vs-true plots, and zone analysis comparing XGBoost against the LR
baseline. Also: top-5 outlier analysis explaining the linear-label NASA
Score explosion.

**Section 5 — Feature importance.** Gain-based importance extracted from
the tuned piecewise model. Cross-checked against EDA degradation-signal
findings.

**Section 6 — Save and finalize.** Per-engine predictions and sweep table
saved; experiment log updated.

## Results (recorded, final run)

| Run | Label | MAE | RMSE | NASA Score | Train time (s) |
|---|---|---|---|---|---|
| EXP-003 | linear | 24.71 | 34.72 | 99568.46 | 1.6568 |
| EXP-004 | piecewise125 | 13.21 | 17.71 | 784.53 | 1.5925 |

**Best config (both labels):** max_depth=8, n_estimators=300,
learning_rate=0.05.

**XGBoost (piecewise) vs. LR (piecewise):**
- MAE: 17.79 → 13.21 (−25.7%)
- RMSE: 22.07 → 17.71 (−19.8%)
- NASA Score: 1334.57 → 784.53 (−41.2%)
- Training time: 0.02 s → 1.59 s (67× more expensive in relative terms;
  still only ~1.6 s in absolute terms)

## Zone analysis — the near-failure bias test

| Zone | n | XGB MAE | XGB bias | LR MAE | LR bias |
|---|---|---|---|---|---|
| Near failure (RUL ≤ 30) | 25 | 10.07 | +8.59 | 19.85 | +18.77 |
| Mid life (30 < RUL ≤ 80) | 20 | 14.83 | +8.83 | 19.76 | +19.75 |
| Early life (RUL > 80) | 55 | 14.04 | −3.21 | 16.14 | −8.46 |

**Hypothesis confirmed:** the near-failure over-prediction bias was
substantially caused by linearity. XGBoost cuts the dangerous +18.77
mean bias to +8.59 (−54%). The remaining +8.59 bias is not eliminated
by non-linearity alone — it is carried forward as a motivation for the
LSTM (Notebook 05), which can additionally exploit the degradation
trajectory rather than a single-row snapshot.

## Feature importance (piecewise model)

| Rank | Sensor | Symbol | Gain share | EDA finding |
|---|---|---|---|---|
| 1 | s11 | Ps30 — HPC static pressure | 48.5% | Strong monotonic trend; HPC fault origin |
| 2 | s4 | T50 — LPT outlet temperature | 17.0% | Strong rising trend with degradation |
| 3 | s9 | Nc — physical core speed | 9.3% | Divergent near failure; absorbs s14 info |
| 4 | s12 | phi — fuel/Ps30 ratio | 5.2% | Trends with degradation |
| 5 | s7 | P30 — HPC outlet pressure | 3.1% | Trends with degradation |
| 6–13 | s15,s13,s17,s20,s21,s8,s3,s2 | — | 25.2% combined | Minor contributors |

Top-3 sensors (s11, s4, s9) account for 74.8% of total importance.
The model reads the HPC fault where it originates (s11 = HPC pressure,
s12 = HPC fuel ratio, s7 = HPC outlet pressure). s9 absorbs the
information of the dropped redundant sensor s14 — confirming the
preprocessing decision.

## Complexity-label interaction — a key secondary finding

With the linear label, XGBoost produces a catastrophic NASA Score of
99,568 — seven times worse than plain Linear Regression (14,094). Five
engines account for 90.2% of this score. Engine 78 alone (true RUL 107,
predicted 217, error +110 cycles) contributes 62,856 to the total.

**Mechanism:** XGBoost commits harder to extreme predictions than LR
(regression to the mean no longer dampens predictions). With a
physically unreasonable label (linear, targets up to 361 cycles), the
model produces very large predictions for engines with high sensor
readings — and when these engines are actually near failure, the
exponential NASA Score penalty explodes.

**Thesis implication:** model complexity amplifies label flaws. More
capacity helps with a good label (piecewise: −41% NASA Score); it
actively hurts with a bad label (linear: +607% NASA Score vs LR).
This is a nuanced, thesis-level finding for the secondary research
question that goes beyond "piecewise is better."

## Strengths (Vorteile)

1. **Substantially better accuracy than LR with the piecewise label.**
   −26% MAE, −20% RMSE, −41% NASA Score. Non-linearity and feature
   interactions pay off.

2. **Interpretable via feature importance.** The model shows which
   sensors drive predictions. This is not possible with LSTM or Chronos.
   The importance ranking independently confirms the EDA: s11, s4, s9
   dominate, consistent with HPC degradation physics.

3. **Halved the dangerous near-failure bias.** Over-prediction of
   nearly-failed engines reduced from +18.77 to +8.59 — a direct safety
   improvement over LR.

4. **Fast and cheap.** ~1.6 s training time. Still essentially free in
   any practical setting, despite being 67× slower than LR.

5. **Robust to moderate tuning choices.** The six sweep configurations
   produced validation RMSEs in a narrow range (25–29), indicating the
   model is not highly sensitive to these hyperparameters.

## Weaknesses (Nachteile)

1. **Complexity amplifies label flaws.** With the linear label, XGBoost
   is dramatically worse than LR on NASA Score (99,568 vs 14,094). Five
   extreme predictions dominate the total score. A more powerful model
   requires a well-designed label — it does not compensate for label
   problems.

2. **Near-failure bias not eliminated.** The +8.59 mean over-prediction
   in the near-failure zone is substantially reduced but not removed.
   The model still does not fully commit to "this engine is about to
   fail." The likely cause: XGBoost sees only a single row per
   prediction — it cannot observe how fast the engine is degrading.
   Two engines with identical current sensor readings but different
   degradation rates have different true RUL, but XGBoost cannot
   distinguish them.

3. **No temporal context.** XGBoost sees one snapshot per prediction,
   not a trajectory. The EDA (Section 11) showed that degradation
   rates vary by 3.3× across engines — identical sensor snapshots
   can correspond to very different remaining lives. A model blind to
   the trajectory cannot fully resolve this.

4. **Not interpretable in time.** Feature importance shows which sensors
   matter across the whole dataset but cannot show how the importance
   changes as the engine approaches failure.

## The precise problem statement for Notebook 05

> XGBoost's remaining core problem: it sees each engine as a single
> snapshot in time, not as a degrading system moving along a trajectory.
> The 3.3× spread in degradation rates (EDA Section 11) means two
> engines with identical current sensor values can have very different
> remaining lives — and XGBoost cannot distinguish them. The LSTM
> (Notebook 05) addresses this by processing the full sequence of
> sensor readings up to the current cycle, allowing it to infer not
> just the current state but also the rate and pattern of degradation.

## Handoff to LSTM (Notebook 05)

| Weakness | Addressed by LSTM? | How |
|---|---|---|
| 1. Label amplification | Partially (still needs piecewise) | Better near-failure predictions reduce extreme outliers |
| 2. Near-failure bias (+8.59) | ✅ Testable hypothesis | Trajectory reading should identify fast-degrading engines |
| 3. No temporal context | ✅ Yes — primary target | LSTM processes sequences: 30-cycle window of sensor history |
| 4. No temporal importance | ✅ Yes | LSTM weights change across time steps implicitly |

**Cost side:** LSTM training requires sequence construction (sliding
window), a neural network with many parameters, and GPU training.
Training time will rise from ~1.6 s to minutes. The key question:
does trajectory reading reduce the remaining near-failure bias enough
to justify the substantially higher cost?

## Ladder so far

| Model | MAE (piecewise) | NASA (piecewise) | Train time |
|---|---|---|---|
| Linear Regression | 17.79 | 1334.57 | 0.02 s |
| XGBoost (tuned) | 13.21 | 784.53 | 1.59 s |
| LSTM | — | — | — |
| Chronos | — | — | — |

**Next:** `05_LSTM.ipynb` — processes full sensor trajectories via a
sliding window; key question: does temporal context reduce the
remaining +8.59 near-failure bias and improve the MAE below 13.21?